# 🚢 Titanic — Análise Exploratória de Dados (EDA)

Este notebook cobre a exploração completa do dataset, respondendo perguntas como:
- Qual a taxa de sobrevivência geral?
- Gênero e classe social influenciaram a sobrevivência?
- Como tratar valores ausentes?
- Quais features usar no modelo?

---
> **Fluxo do projeto:** `01_EDA` → `02_Logistic_Regression` → `03_Random_Forest` → `04_Voting_Classifier`

## 1. Importações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.preprocessing import OneHotEncoder

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

## 2. Carregamento dos dados

In [ ]:
df = pd.read_csv('dataset/titanicc/train.csv')

print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Análise de Sobrevivência

### 3.1 Visão Geral

In [ ]:
total       = len(df)
mortos      = (df['Survived'] == 0).sum()
vivos       = (df['Survived'] == 1).sum()

print(f"Total de passageiros : {total}")
print(f"Sobreviventes        : {vivos}  ({vivos/total*100:.1f}%)")
print(f"Mortos               : {mortos} ({mortos/total*100:.1f}%)")

# Gráfico de pizza
fig, ax = plt.subplots()
ax.pie([vivos, mortos], labels=['Sobreviveu', 'Morreu'],
       autopct='%1.1f%%', colors=['#4CAF50', '#F44336'],
       startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax.set_title('Distribuição de Sobrevivência', fontsize=14)
plt.show()

### 3.2 Sobrevivência por Sexo e Classe

In [ ]:
# Taxa de sobrevivência por sexo
survival_sex = df.groupby('Sex')['Survived'].agg(['sum', 'count'])
survival_sex.columns = ['sobreviventes', 'total']
survival_sex['taxa_%'] = (survival_sex['sobreviventes'] / survival_sex['total'] * 100).round(2)
print("\n--- Por Sexo ---")
print(survival_sex)

# Taxa por classe
survival_class = df.groupby('Pclass')['Survived'].agg(['sum', 'count'])
survival_class.columns = ['sobreviventes', 'total']
survival_class['taxa_%'] = (survival_class['sobreviventes'] / survival_class['total'] * 100).round(2)
print("\n--- Por Classe ---")
print(survival_class)

In [ ]:
# Gráfico: taxa de sobrevivência por sexo e classe
survival_rate = df.groupby(['Pclass', 'Sex'])['Survived'].mean().reset_index()
survival_rate['Survived'] *= 100

fig, ax = plt.subplots()
sns.barplot(x='Pclass', y='Survived', hue='Sex', data=survival_rate, ax=ax)

ax.set_title('Taxa de Sobrevivência por Classe e Sexo', fontsize=14)
ax.set_xlabel('Classe')
ax.set_ylabel('Taxa de Sobrevivência (%)')
ax.set_ylim(0, 115)
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['1ª Classe', '2ª Classe', '3ª Classe'])

for p in ax.patches:
    ax.text(p.get_x() + p.get_width() / 2., p.get_height() + 1.5,
            f'{p.get_height():.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

### 3.3 Sobrevivência por Porto de Embarque

In [ ]:
survival_emb = df.groupby('Embarked')['Survived'].agg(['sum', 'count'])
survival_emb.columns = ['sobreviventes', 'total']
survival_emb['taxa_%'] = (survival_emb['sobreviventes'] / survival_emb['total'] * 100).round(2)
print(survival_emb)

# Mapa dos portos
portos = {'S': 'Southampton (S)', 'C': 'Cherbourg (C)', 'Q': 'Queenstown (Q)'}
print("\nLegenda:", portos)

### 3.4 Sobrevivência por Faixa Etária

In [ ]:
bins   = [0, 2, 12, 18, 30, 60, 120, np.inf]
labels = ['recém-nascido', 'criança', 'adolescente', 'jovem', 'adulto', 'idoso', 'Não_Informado']
df['faixa_etaria'] = pd.cut(df['Age'], bins=bins, labels=labels, right=False)
df['faixa_etaria'] = df['faixa_etaria'].fillna('Não_Informado')

survival_faixa = df.groupby('faixa_etaria', observed=False)['Survived'].agg(['sum', 'count'])
survival_faixa.columns = ['sobreviventes', 'total']
survival_faixa['taxa_%'] = (survival_faixa['sobreviventes'] / survival_faixa['total'] * 100).round(2)
print(survival_faixa)

# Gráfico
fig, ax = plt.subplots()
survival_faixa['taxa_%'].plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Taxa de Sobrevivência por Faixa Etária', fontsize=14)
ax.set_ylabel('Taxa (%)')
ax.set_xlabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 4. Análise de Valores Ausentes

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'ausentes': missing, '%': missing_pct})
missing_df = missing_df[missing_df['ausentes'] > 0].sort_values('%', ascending=False)
print(missing_df)

# Mapa de calor de nulos
fig, ax = plt.subplots(figsize=(12, 3))
sns.heatmap(df.isnull().T, cbar=False, cmap='viridis', ax=ax)
ax.set_title('Mapa de Valores Ausentes', fontsize=14)
plt.tight_layout()
plt.show()

**Decisões de tratamento:**
- `Cabin` → ~77% nulo → **remover** a coluna
- `Age` → ~20% nulo → **discretizar** em faixa etária; nulos viram `Não_Informado`
- `Embarked` → 2 nulos → **preencher** com moda (`'S'`)

## 5. Distribuição das Features Numéricas

In [ ]:
num_cols = ['Age', 'Fare', 'SibSp', 'Parch']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flatten(), num_cols):
    df[col].hist(bins=30, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(col)
    ax.set_ylabel('Frequência')
plt.suptitle('Distribuição das Features Numéricas', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6. Categorização de Fare (base para o Pipeline)

> Os bins foram definidos com base nos quartis do dataset de treino para evitar data leakage.

In [ ]:
print(df['Fare'].describe())

bins_fare   = [0, 7.91, 14.45, 31.00, df['Fare'].max()]
labels_fare = ['muito_baixa', 'baixa', 'média', 'alta']
df['Fare_categoria'] = pd.cut(df['Fare'], bins=bins_fare, labels=labels_fare, include_lowest=True)

surv_fare = df.groupby('Fare_categoria', observed=False)['Survived'].mean() * 100
surv_fare.plot(kind='bar', color='coral', edgecolor='white')
plt.title('Taxa de Sobrevivência por Categoria de Tarifa', fontsize=14)
plt.ylabel('Taxa (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Matriz de Correlação

In [ ]:
df_corr = df[['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']].copy()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df_corr.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax)
ax.set_title('Matriz de Correlação', fontsize=14)
plt.tight_layout()
plt.show()

---
## ✅ Conclusões da EDA

| Achado | Impacto no modelo |
|---|---|
| Mulheres sobreviveram muito mais (~74% vs ~19%) | `Sex` é a feature mais discriminante |
| 1ª classe sobreviveu mais (~63%) | `Pclass` é feature importante |
| Crianças tiveram maior sobrevivência | Discretizar `Age` é útil |
| Tarifas altas correlacionam com sobrevivência | Discretizar `Fare` em quartis |
| `Cabin` tem ~77% nulo | Remover a coluna |
| `Age` tem ~20% nulo | Usar faixas + categoria `Não_Informado` |
| `Name`, `Ticket`, `PassengerId` | Sem valor preditivo → remover |

**Próximo passo:** `02_Logistic_Regression.ipynb`